In [0]:
from pyspark.sql import functions as F

# Widget for source path
dbutils.widgets.text(
    "source_path",
    "/Volumes/workspace/ecommerce/ecommerce_data/",
    "Source Path"
)

source_path = dbutils.widgets.get("source_path")

bronze_path = "/Volumes/workspace/ecommerce/ecommerce_data/medallion/bronze/events"

# Read raw data
events_raw = spark.read.csv(
    source_path,
    header=True,
    inferSchema=True
)

# Write to Bronze (raw ingestion)
events_raw.write \
    .format("delta") \
    .mode("append") \
    .save(bronze_path)

print("✅ Bronze layer completed")


In [0]:
from pyspark.sql import functions as F

bronze_path = "/Volumes/workspace/ecommerce/ecommerce_data/medallion/bronze/events"
silver_path = "/Volumes/workspace/ecommerce/ecommerce_data/medallion/silver/events"

# Read from Bronze
bronze_df = spark.read.format("delta").load(bronze_path)

# Clean & validate
silver_df = (
    bronze_df
    .filter(F.col("event_type").isNotNull())
    .filter(F.col("price").isNotNull())
    .filter(F.col("user_id").isNotNull())
    .dropDuplicates(["user_session", "event_time"])
)

# Write to Silver
silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(silver_path)

print("✅ Silver layer completed")


In [0]:
from pyspark.sql import functions as F

silver_path = "/Volumes/workspace/ecommerce/ecommerce_data/medallion/silver/events"
gold_path = "/Volumes/workspace/ecommerce/ecommerce_data/medallion/gold"

silver_df = spark.read.format("delta").load(silver_path)

# Gold table 1: Brand Revenue
brand_revenue = (
    silver_df
    .filter(F.col("event_type") == "purchase")
    .groupBy("brand")
    .agg(F.sum("price").alias("total_revenue"))
)

brand_revenue.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{gold_path}/brand_revenue")

# Gold table 2: Conversion Rate
conversion_rate = (
    silver_df
    .groupBy("category_code", "event_type")
    .count()
    .groupBy("category_code")
    .pivot("event_type")
    .sum("count")
    .withColumn(
        "conversion_rate",
        (F.col("purchase") / F.col("view")) * 100
    )
)

conversion_rate.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{gold_path}/conversion_rate")

print("✅ Gold layer completed")
